In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import sys

sys.path.append("../")
from torch_src.model.components import (
    PoolerTransformerBlock,
    TransformerBlock,
    get_mlp,
)


In [2]:
import numpy as np
def sample_circle(n,  dim=2, noise=0):
    points = np.full((n, dim), -1, dtype=np.float32)
    sample_points = np.random.randn(n, dim)
    sample_points /= np.linalg.norm(sample_points, axis=1, keepdims=True)
    r = np.random.uniform(0.5, 1, size=dim)
   # sample_points *= r  # Scale by random radius
    if noise > 0:
        noise_points = np.random.normal(0, 1, sample_points.shape) * noise * np.min(r)
        sample_points += noise_points
    points[:n, :] = sample_points
    return points


def sample_square(n, dim=2, noise=0):
    points = np.full((n, dim), -1, dtype=np.float32)
    sample_points = np.random.uniform(-1, 1, (n, dim))
    sample_points /= np.linalg.norm(sample_points, axis=1, keepdims=True, ord=np.inf)
    r = np.random.uniform(0.5, 1, size=dim)
    #sample_points *= r  # Scale by random radius
    if noise > 0:
        noise_points = np.random.normal(0, 1, sample_points.shape) * noise * np.min(r)
        sample_points += noise_points
    points[:n, :] = sample_points
    return points

# -------------------------------
# Synthetic dataset
# -------------------------------
class GraphSetDataset(Dataset):
    """
    Simple dataset of variable-size graphs.
    Each graph has random nodes and a binary label.
    """
    def __init__(self, num_graphs=10000, max_nodes=20, embed_dim=2):
        self.graphs = []
        self.labels = []
        for _ in range(num_graphs):
            num_nodes = torch.randint(10, max_nodes+1, (1,)).item()
            y = torch.randint(0, 2, (1,)).item()
            if y == 0:
                nodes = torch.tensor(sample_circle(num_nodes, embed_dim), dtype=torch.float32)
            else:
                nodes = torch.tensor(sample_square(num_nodes, embed_dim), dtype=torch.float32)
            self.graphs.append(nodes)
            self.labels.append(y)
    
    def __len__(self):
        return len(self.graphs)
    
    def __getitem__(self, idx):
        x = self.graphs[idx]       # [num_nodes, embed_dim]
        y = self.labels[idx]       # int
        return x, y


def collate_fn(batch):
    """
    Build minibatch of variable-size graphs.
    - x: concatenated nodes [N_total, embed_dim]
    - batch_idx: graph membership [N_total]
    - y: graph labels [B]
    """
    x_list, y_list = zip(*batch)

    x = torch.cat(x_list, dim=0)  # all nodes together
    # assign batch indices 0..B-1
    batch_idx = torch.cat([
        torch.full((nodes.size(0),), i, dtype=torch.long)
        for i, nodes in enumerate(x_list)
    ])
    y = torch.tensor(y_list, dtype=torch.long)
    return x, batch_idx, y



In [3]:

# -------------------------------
# Simple model
# -------------------------------
from torch_scatter import scatter_mean, scatter_max, scatter_sum
class GraphClassifier(nn.Module):
    def __init__(self,input_dim = 2, embed_dim=8, num_heads=2, num_seeds=1):
        super().__init__()
        self.input_dim = input_dim
        self.embed_dim = embed_dim

        self.input_mlp = get_mlp(input_dim, embed_dim, 2)
        self.attn_1 = TransformerBlock(embed_dim, num_heads)
        self.attn_2 = TransformerBlock(embed_dim, num_heads)
        self.pool = PoolerTransformerBlock(embed_dim, num_heads, num_seeds)
        self.fc = get_mlp(embed_dim, 2, 4)

    def forward(self, x, batch_idx):
        x = self.input_mlp(x)
        x = self.attn_1(x, batch_idx)
        x = self.attn_2(x, batch_idx)
        pooled = self.pool(x, batch_idx)  #  # (B, embed_dim)
         # (B, embed_dim)
        logits = self.fc(pooled)
        return nn.functional.softmax(logits, dim=-1)  # (B, num_classes)

# -------------------------------
# Training setup
# -------------------------------
dataset = GraphSetDataset(num_graphs=10000)
loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GraphClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# -------------------------------
# Training loop
# -------------------------------
for epoch in range(10):
    model.train()
    total_loss = 0
    for x, batch_idx, y in tqdm(loader, desc=f"Epoch {epoch+1} [Train]"):
        x, batch_idx, y = x.to(device), batch_idx.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x, batch_idx)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataset):.4f}")

# -------------------------------
# Simple evaluation
# -------------------------------
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for x, batch_idx, y in loader:
        x, batch_idx, y = x.to(device), batch_idx.to(device), y.to(device)
        logits = model(x, batch_idx)
        pred = logits.argmax(dim=-1)
        correct += (pred == y).sum().item()
        total += y.size(0)
print("Accuracy:", correct/total)


Epoch 1 [Train]: 100%|██████████| 313/313 [00:05<00:00, 57.93it/s]


Epoch 1, Loss: 10.3620


Epoch 2 [Train]: 100%|██████████| 313/313 [00:05<00:00, 57.83it/s]


Epoch 2, Loss: 9.4991


Epoch 3 [Train]: 100%|██████████| 313/313 [00:05<00:00, 56.67it/s]


...

Epoch 8, Loss: 5.8318


Epoch 9 [Train]: 100%|██████████| 313/313 [00:05<00:00, 57.12it/s]


Epoch 9, Loss: 5.8285


Epoch 10 [Train]: 100%|██████████| 313/313 [00:05<00:00, 56.73it/s]


Epoch 10, Loss: 5.7327
Accuracy: 0.939
